# Algorithmic Systems Design: Smart Spell-Checker & Autocorrect Pipeline

**Authors:** Jakub Habib, Ulugbek Tojiboev

---

### System Objective

A real-time pipeline that verifies user text input and suggests the **top 3** statistically probable corrections for typos. The system chains three algorithmic stages:

| Stage | Role | Data Structure |
|:------|:-----|:---------------|
| 1. Validation | Instant dictionary lookup | Chaining Hash Table — average $O(1)$ |
| 2. Correction | Find similar words (Levenshtein distance) | Space-optimized DP — $O(N \times M)$ time, $O(M)$ space |
| 3. Ranking | Return top-*k* suggestions by frequency | Min-Heap — $O(C \log k)$ |

## Stage 1 — Vocabulary Validation (Custom Chaining Hash Table)

The `ChainingHashTable` stores known words using modulo hashing and separate chaining. Word validation is performed through `Search()` with average-case $O(1)$ lookup time.

In [ ]:
import re

class DataPreparer:
    @staticmethod
    def clean_text(text):
        if not text:
            return ""
        return re.sub(r'[^a-zA-Z]', '', text).lower()

    @staticmethod
    def load_database(raw_data, hash_table):
        for word, freq in raw_data:
            cleaned = DataPreparer.clean_text(word)
            if cleaned:
                hash_table.Add(cleaned, freq)

class ChainingHashTable:
    """Stage 1: Hash Table with separate chaining and modulo hashing."""
    def Init(self, p=100003):
        self.p = p
        self.table = [[] for _ in range(self.p)]

    def _hash_func(self, k_str):
        hash_val = 0
        for char in k_str:
            hash_val = (hash_val * 31 + ord(char)) % self.p
        return hash_val

    def Add(self, k, v):
        idx = self._hash_func(k)
        for i, (key, val) in enumerate(self.table[idx]):
            if key == k:
                self.table[idx][i] = (k, v)
                return
        self.table[idx].append((k, v))

    def Search(self, k):
        idx = self._hash_func(k)
        for key, val in self.table[idx]:
            if key == k:
                return True
        return False

    def GetValue(self, k):
        idx = self._hash_func(k)
        for key, val in self.table[idx]:
            if key == k:
                return val
        return 0

    def GetAllKeys(self):
        keys = []
        for bucket in self.table:
            for k, v in bucket:
                keys.append(k)
        return keys

# --- Demo ---
mock_database = [("the", 100000), ("tea", 500), ("ten", 3000), ("hello", 50000), ("definitely", 20000)]
vocab_table = ChainingHashTable()
vocab_table.Init()
DataPreparer.load_database(mock_database, vocab_table)

print(f"Is 'Hello!' valid?   {vocab_table.Search(DataPreparer.clean_text('Hello!'))}")
print(f"Is 'definetly' valid? {vocab_table.Search(DataPreparer.clean_text('definetly'))}")

## Stage 3 — Top-K Ranking (Min-Heap)

Given a set of candidate corrections from Stage 2, the `TopKMinHeap` selects the **k most frequent** words using a min-heap of fixed size *k*. This avoids sorting the entire candidate list and runs in $O(C \log k)$ time.

In [ ]:
import heapq

class TopKMinHeap:
    """Stage 3: Min-Heap structure using custom method naming."""
    def Init(self, k=3):
        self.k = k
        self.heap = []

    def Add(self, priority, value):
        heapq.heappush(self.heap, (priority, value))
        if len(self.heap) > self.k:
            self.RemoveMin()

    def RemoveMin(self):
        return heapq.heappop(self.heap)

    def GetSortedResults(self):
        result = sorted(self.heap, key=lambda x: x[0], reverse=True)
        return [val for priority, val in result]

# --- Demo ---
mock_frequencies = {"tea": 500, "ten": 3000, "the": 100000, "ted": 150, "tech": 15000}
mock_candidates = ["tea", "ten", "the", "ted", "tech"]

ranker = TopKMinHeap()
ranker.Init(k=3)

for word in mock_candidates:
    ranker.Add(mock_frequencies.get(word, 0), word)

print(f"Raw Candidates:    {mock_candidates}")
print(f"Top 3 Suggestions: {ranker.GetSortedResults()}")

## Technical Analysis

### Data Flow

1. The user types a word (e.g. `"teh"`).
2. **Stage 1** — Custom hash table lookup (`ChainingHashTable.Search`). If the word is not found, it is flagged as a typo.
3. **Stage 2** — A length pre-filter discards words whose length differs by more than 2. Levenshtein distance is computed for the remaining words.
4. **Stage 3** — Candidates are ranked by frequency via `TopKMinHeap`; the top 3 are returned.

---

### Complexity Summary

| Stage | Role | Data Structure | Time | Space |
|:------|:-----|:---------------|:-----|:------|
| **1. Validation** | Check spelling | Chaining Hash Table | Average $O(1)$ | $O(V)$ |
| **2. Correction** | Find similar words | Optimized DP (Levenshtein) | $O(N \times M)$ | $O(M)$ |
| **3. Ranking** | Select top-*k* results | Min-Heap (size *k*) | $O(C \log k)$ | $O(k)$ |

> *V* = dictionary size, *N* & *M* = word lengths, *C* = candidate count, *k* = 3.

---

### Design Decisions

**Custom Chaining Hash Table vs. Binary Search Tree (Stage 1)**
A BST requires $O(\log V)$ comparisons. The custom hash table uses modulo hashing with separate chaining and provides average-case $O(1)$ lookup — ideal for fast validation.

**Min-Heap vs. Full Sort (Stage 3)**
Sorting the full candidate list costs $O(C \log C)$. A min-heap capped at size *k* = 3 processes all candidates in $O(C \log k)$ time with bounded memory.

**System Bottleneck — Stage 2**
Levenshtein remains the computational bottleneck in time. The implementation reduces memory from $O(N \times M)$ to $O(M)$ by keeping only two rows of the DP table, while preserving correctness.

In [ ]:
class SpellCheckerPipeline:
    def __init__(self, hash_table):
        self.dictionary = hash_table

    def _levenshtein_distance_optimized(self, s1, s2):
        """SPACE OPTIMIZED: Reduced from O(N*M) to O(M)."""
        n, m = len(s1), len(s2)
        prev_row = [j for j in range(m + 1)]
        curr_row = [0] * (m + 1)

        for i in range(1, n + 1):
            curr_row[0] = i
            for j in range(1, m + 1):
                cost = 0 if s1[i - 1] == s2[j - 1] else 1
                curr_row[j] = min(
                    prev_row[j] + 1,
                    curr_row[j - 1] + 1,
                    prev_row[j - 1] + cost
                )
            prev_row = curr_row.copy()

        return curr_row[m]

    def process_word(self, input_word, ranker):
        normalized_word = DataPreparer.clean_text(input_word)
        if not normalized_word:
            return "Invalid input"

        # Stage 1: Hash Table Lookup
        if self.dictionary.Search(normalized_word):
            return [normalized_word]

        # Stage 2: Candidate Generation
        typo_len = len(normalized_word)

        for dict_word in self.dictionary.GetAllKeys():
            if abs(len(dict_word) - typo_len) <= 2:
                dist = self._levenshtein_distance_optimized(normalized_word, dict_word)
                if dist <= 2:
                    # Stage 3: Min-Heap Integration
                    freq = self.dictionary.GetValue(dict_word)
                    ranker.Add(freq, dict_word)

        return ranker.GetSortedResults()

# --- Full Pipeline Demo ---
mock_vocabulary = [
    ("the", 5000), ("then", 1200), ("tea", 450), ("ten", 300),
    ("there", 2500), ("apple", 800), ("banana", 600), ("to", 4000)
]

# 1. Setup Hash Table
main_table = ChainingHashTable()
main_table.Init()
DataPreparer.load_database(mock_vocabulary, main_table)

# 2. Setup Pipeline
pipeline = SpellCheckerPipeline(main_table)

# 3. Test Cases
ranker1 = TopKMinHeap()
ranker1.Init()
print("Result for 'apple':", pipeline.process_word("apple", ranker1))

ranker2 = TopKMinHeap()
ranker2.Init()
print("Result for 'teh':", pipeline.process_word("teh", ranker2))